# 04_OOP — 13_Encapsulation_and_Abstraction

## Learning Goal

This notebook covers two major OOP concepts:

- **Encapsulation** — controlling access to internal state and behavior.
- **Abstraction** — hiding unnecessary implementation details and exposing an essential interface.

This notebook builds on `08_Properties_and_Getters_Setters.ipynb` rather than reteaching properties from scratch.

## OOP Progression

```text
11 Inheritance
       ↓
12 Polymorphism
       ↓
13 Encapsulation & Abstraction  ← NOW
       ↓
14 Dunder Methods
       ↓
15 References & Mutability
```

# 1. Introduction

### Encapsulation

> Bundling data and the methods that operate on that data, while controlling how the data is accessed or modified.

### Abstraction

> Hiding unnecessary implementation details and exposing only the essential interface.

Mental model:

```text
Encapsulation
    ↓
Protect / control data and behavior

Abstraction
    ↓
Hide implementation details
    ↓
Expose what the user needs
```

# 2. Encapsulation vs Abstraction

| Encapsulation | Abstraction |
|---|---|
| Controls access to data/behavior | Hides implementation details |
| Focuses on protection and organization | Focuses on what an object exposes |
| Uses conventions, properties, methods, name mangling | Uses abstract classes and interfaces |
| "How do I control access?" | "What should the user need to know?" |

Example:

```text
BankAccount
    ↓
balance is controlled
    ↓
deposit()
withdraw()
```

# 3. Why Encapsulation Is Needed

Without controlled access, an object's state can easily become invalid.

In [ ]:
class BankAccount:
    def __init__(self, balance):
        self.balance = balance


account = BankAccount(5000)
account.balance = -100000

print(account.balance)

The class cannot prevent the caller from assigning an invalid balance.

A better design keeps the internal state controlled.

In [ ]:
class BankAccount:
    def __init__(self, balance):
        self._balance = balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Amount must be positive.")
        self._balance += amount


account = BankAccount(5000)
account.deposit(1000)

print(account._balance)

The important idea is not simply hiding the variable. The class controls **how its state changes**.

# 4. Encapsulation in Python

Python does not enforce private access in exactly the same way as languages such as Java or C++.

Python mainly uses conventions and mechanisms:

```text
public
protected convention
private / name mangling
```

# 5. Public Members

Public members have no leading underscore.

In [ ]:
class Student:
    def __init__(self, name):
        self.name = name

    def display(self):
        print(self.name)


student = Student("Komal")

print(student.name)
student.display()

Public members are intended to be part of the normal interface of the class.

# 6. Protected Convention — `_name`

A single leading underscore communicates:

> This member is intended for internal or protected use.

It is a **convention**, not strict access control.

In [ ]:
class BankAccount:
    def __init__(self, balance):
        self._balance = balance


account = BankAccount(5000)

# Technically possible, but conventionally treated as internal.
print(account._balance)

```text
_name
  ↓
Convention
  ↓
"Please treat this as internal"
```

# 7. Private Members — `__name`

Two leading underscores trigger **name mangling**.

In [ ]:
class BankAccount:
    def __init__(self, balance):
        self.__balance = balance


account = BankAccount(5000)

# account.__balance would normally raise AttributeError.

Python internally changes the name approximately to:

```text
_BankAccount__balance
```

This mechanism is called **name mangling**.

# 8. Name Mangling

Consider:

In [ ]:
class Student:
    def __init__(self):
        self.__marks = 90


student = Student()

try:
    print(student.__marks)
except AttributeError as e:
    print(type(e).__name__)

The mangled name can technically be accessed:

```python
_Student__marks
```

Name mangling is not true security or absolute privacy.

In [ ]:
class Student:
    def __init__(self):
        self.__marks = 90


student = Student()

print(student._Student__marks)

Name mangling mainly helps to:

- Avoid accidental access
- Avoid accidental overriding
- Prevent naming conflicts in inheritance

# 9. Why Name Mangling Exists

Consider inheritance.

In [ ]:
class Parent:
    def __init__(self):
        self.__value = "Parent"


class Child(Parent):
    def __init__(self):
        super().__init__()
        self.__value = "Child"


child = Child()

print(child._Parent__value)
print(child._Child__value)

The two attributes become approximately:

```text
_Parent__value
_Child__value
```

They do not accidentally collide.

This is one useful reason for double underscores.

# 10. Public vs Protected vs Private

```python
class Example:
    def __init__(self):
        self.public = 1
        self._protected = 2
        self.__private = 3
```

| Syntax | Meaning |
|---|---|
| `public` | Public |
| `_protected` | Internal/protected convention |
| `__private` | Name-mangled attribute |

`_protected` and `__private` are not identical mechanisms.

# 11. Encapsulation Through Methods

Instead of directly changing internal data:

```python
account.balance = -5000
```

provide meaningful operations:

```python
account.deposit(1000)
account.withdraw(500)
```

In [ ]:
class BankAccount:
    def __init__(self, balance):
        self._balance = balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Invalid amount.")
        self._balance += amount

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("Invalid amount.")
        if amount > self._balance:
            raise ValueError("Insufficient balance.")
        self._balance -= amount


account = BankAccount(5000)
account.deposit(1000)
account.withdraw(500)

print(account._balance)

The class now controls valid state transitions.

# 12. Encapsulation Through Properties

You already learned `@property` in notebook 08. Here, focus on its **role in encapsulation**.

A property lets the class expose a convenient public interface while keeping validation and storage under its control.

In [ ]:
class Student:
    def __init__(self, marks):
        self.marks = marks

    @property
    def marks(self):
        return self._marks

    @marks.setter
    def marks(self, value):
        if not 0 <= value <= 100:
            raise ValueError("Marks must be between 0 and 100.")
        self._marks = value


student = Student(85)
print(student.marks)

student.marks = 90
print(student.marks)

The caller uses:

```python
student.marks = 90
```

while the class controls validation internally.

# 13. Encapsulation Does Not Mean "Hide Everything"

A common misconception is:

> Encapsulation means making every variable private.

Instead:

> Design a clear interface and control how internal state is accessed or modified.

A useful design looks like:

```text
Public interface
    ↓
deposit()
withdraw()
balance
    ↓
Internal state
_balance
```

The user interacts with meaningful operations rather than manipulating implementation details directly.

# 14. Read-Only Data

A property without a setter can provide controlled read-only access.

In [ ]:
class Employee:
    def __init__(self, employee_id):
        self._employee_id = employee_id

    @property
    def employee_id(self):
        return self._employee_id


employee = Employee(101)

print(employee.employee_id)

try:
    employee.employee_id = 202
except AttributeError as e:
    print(type(e).__name__)

There is no setter, so the public property cannot be assigned normally.

# 15. Encapsulation and Object Validity

A well-designed object should protect its own valid state.

In [ ]:
class Temperature:
    def __init__(self, value):
        self.temperature = value

    @property
    def temperature(self):
        return self._temperature

    @temperature.setter
    def temperature(self, value):
        if value < -273.15:
            raise ValueError(
                "Temperature cannot be below absolute zero."
            )
        self._temperature = value


temperature = Temperature(25)
print(temperature.temperature)

temperature.temperature = 30
print(temperature.temperature)

Invalid state is rejected by the class itself.

# 16. What Is Abstraction?

Abstraction means hiding unnecessary implementation details and exposing a useful interface.

For example, when you use:

```python
print("Hello")
```

you do not need to know all of the internal details of output handling.

You use a simple interface:

```text
print()
```

The implementation is hidden behind that interface.

# 17. Real-World Abstraction Examples

### ATM

You see:

```text
Withdraw
Deposit
Check Balance
```

You do not need to see:

```text
Database queries
Authentication mechanisms
Bank network communication
Transaction processing
```

### Car

You use:

```text
Start
Accelerate
Brake
```

without needing to know every internal engine operation.

### Data Science

You may write:

```python
model.fit(X_train, y_train)
```

without manually implementing every optimization step.

These are examples of abstraction.

# 18. Abstraction in Python

Python supports formal abstraction through:

- Abstract classes
- Abstract methods
- The `abc` module

Start with:

```python
from abc import ABC, abstractmethod
```

# 19. Abstract Base Class — `ABC`

`ABC` stands for **Abstract Base Class**.

It is used to define a base class intended to provide a common interface or contract.

In [ ]:
from abc import ABC


class Animal(ABC):
    pass


print(Animal)

An `ABC` can be combined with `@abstractmethod` to require subclasses to implement particular behavior.

# 20. Abstract Methods

An abstract method defines behavior that concrete subclasses are expected to implement.

In [ ]:
from abc import ABC, abstractmethod


class Animal(ABC):
    @abstractmethod
    def sound(self):
        pass

The class now defines a contract:

```text
Animal
  ↓
must provide sound()
```

# 21. Implementing an Abstract Class

In [ ]:
from abc import ABC, abstractmethod


class Animal(ABC):
    @abstractmethod
    def sound(self):
        pass


class Dog(Animal):
    def sound(self):
        print("Bark")


dog = Dog()
dog.sound()

The child implements the required method, so `Dog` can be instantiated.

# 22. Cannot Instantiate an Incomplete Abstract Class

If the abstract method has not been implemented, the abstract class cannot be instantiated.

In [ ]:
from abc import ABC, abstractmethod


class Animal(ABC):
    @abstractmethod
    def sound(self):
        pass


try:
    animal = Animal()
except TypeError as e:
    print(type(e).__name__)
    print(e)

The important pattern is:

```text
Abstract class
      ↓
Defines required behavior
      ↓
Child class implements it
      ↓
Concrete object can be created
```

# 23. Abstract Class as a Contract

Consider a shape hierarchy.

In [ ]:
from abc import ABC, abstractmethod


class Shape(ABC):
    @abstractmethod
    def area(self):
        pass


class Rectangle(Shape):
    def __init__(self, length, width):
        self.length = length
        self.width = width

    def area(self):
        return self.length * self.width


class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return 3.14159 * self.radius ** 2


shapes = [
    Rectangle(10, 5),
    Circle(7)
]

for shape in shapes:
    print(shape.area())

The parent defines the **contract** and the child classes provide the **implementation**.

# 24. Abstract Class with Common Implementation

An abstract class can contain normal concrete methods as well as abstract methods.

In [ ]:
from abc import ABC, abstractmethod


class Animal(ABC):
    @abstractmethod
    def sound(self):
        pass

    def sleep(self):
        print("Animal is sleeping")


class Dog(Animal):
    def sound(self):
        print("Bark")


dog = Dog()

dog.sound()
dog.sleep()

So an abstract class can contain:

```text
Abstract methods
        +
Concrete methods
```

# 25. Abstract Class with Attributes

Abstract classes can also have constructors and normal attributes.

In [ ]:
from abc import ABC, abstractmethod


class Employee(ABC):
    def __init__(self, name):
        self.name = name

    @abstractmethod
    def calculate_salary(self):
        pass


class Developer(Employee):
    def calculate_salary(self):
        return 60000


developer = Developer("Komal")

print(developer.name)
print(developer.calculate_salary())

An abstract class is still a normal class in many respects; its key difference is that it can require subclasses to implement abstract behavior.

# 26. Multiple Abstract Methods

A class can define more than one required operation.

In [ ]:
from abc import ABC, abstractmethod


class Payment(ABC):
    @abstractmethod
    def pay(self, amount):
        pass

    @abstractmethod
    def refund(self, amount):
        pass


class UPI(Payment):
    def pay(self, amount):
        print(f"Paid ₹{amount} using UPI")

    def refund(self, amount):
        print(f"Refunded ₹{amount} through UPI")


payment = UPI()
payment.pay(1000)
payment.refund(200)

A concrete subclass must implement all required abstract methods.

# 27. Partial Implementation

If a subclass implements only one of several abstract methods, it remains abstract.

In [ ]:
from abc import ABC, abstractmethod


class Payment(ABC):
    @abstractmethod
    def pay(self, amount):
        pass

    @abstractmethod
    def refund(self, amount):
        pass


class UPI(Payment):
    def pay(self, amount):
        print("Paid")


try:
    upi = UPI()
except TypeError as e:
    print(type(e).__name__)
    print(e)

`refund()` is still abstract, so `UPI` cannot yet be instantiated.

# 28. Abstract Classes + Polymorphism

Abstraction and polymorphism work naturally together.

The abstract parent defines the common interface. Concrete subclasses provide different implementations.

In [ ]:
from abc import ABC, abstractmethod


class Shape(ABC):
    @abstractmethod
    def area(self):
        pass


class Rectangle(Shape):
    def area(self):
        return 100


class Circle(Shape):
    def area(self):
        return 50


shapes = [
    Rectangle(),
    Circle()
]

for shape in shapes:
    print(shape.area())

The relationship is:

```text
Abstraction
     ↓
Common interface
     ↓
Polymorphism
     ↓
Different implementations
```

This connects directly to notebook 12.

# 29. Encapsulation vs Abstraction

### Encapsulation

```text
How do I control access
to internal state?
```

Examples:

```python
self._balance
deposit()
withdraw()
```

### Abstraction

```text
What should the user
need to know?
```

Example:

```python
payment.pay(1000)
```

The user does not need to know the internal payment-processing implementation.

# 30. Encapsulation vs Abstraction vs Inheritance vs Polymorphism

| Concept | Main idea | Example |
|---|---|---|
| Encapsulation | Control internal state/access | `BankAccount._balance` |
| Abstraction | Hide implementation details | `Payment.pay()` |
| Inheritance | Reuse/extend classes | `Dog(Animal)` |
| Polymorphism | Same interface, different behavior | `dog.speak()` / `cat.speak()` |

Mental model:

```text
Inheritance
→ What can I reuse?

Polymorphism
→ How can different objects respond to the same interface?

Encapsulation
→ How do I protect/control internal state?

Abstraction
→ What implementation details can I hide?
```

# 31. Practical Project — Bank Account

This project combines encapsulation, controlled access, validation, and a public interface.

In [ ]:
class BankAccount:
    def __init__(self, balance):
        if balance < 0:
            raise ValueError("Initial balance cannot be negative.")
        self._balance = balance

    @property
    def balance(self):
        return self._balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Invalid amount.")
        self._balance += amount

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("Invalid amount.")
        if amount > self._balance:
            raise ValueError("Insufficient balance.")
        self._balance -= amount


account = BankAccount(5000)

account.deposit(1000)
account.withdraw(500)

print(account.balance)

The caller uses the public interface while the class controls the internal balance.

# 32. Practical Project — Payment Abstraction

Create an abstract payment interface.

In [ ]:
from abc import ABC, abstractmethod


class Payment(ABC):
    @abstractmethod
    def pay(self, amount):
        pass


class UPI(Payment):
    def pay(self, amount):
        print(f"Paid ₹{amount} using UPI")


class CreditCard(Payment):
    def pay(self, amount):
        print(f"Paid ₹{amount} using credit card")


payments = [
    UPI(),
    CreditCard()
]

for payment in payments:
    payment.pay(1000)

This combines:

```text
Abstraction
    +
Inheritance
    +
Polymorphism
```

# 33. Practical Project — Data Processor

This example is especially relevant to data-processing and Data Science workflows.

Define a common processor contract.

In [ ]:
from abc import ABC, abstractmethod


class DataProcessor(ABC):
    @abstractmethod
    def load(self):
        pass

    @abstractmethod
    def process(self):
        pass


class CSVProcessor(DataProcessor):
    def load(self):
        print("Loading CSV")

    def process(self):
        print("Processing CSV")


class JSONProcessor(DataProcessor):
    def load(self):
        print("Loading JSON")

    def process(self):
        print("Processing JSON")


processors = [
    CSVProcessor(),
    JSONProcessor()
]

for processor in processors:
    processor.load()
    processor.process()

The common interface lets a processing pipeline work with different data-source implementations.

# 34. Practical Project — Employee System

Create an abstract employee contract.

In [ ]:
from abc import ABC, abstractmethod


class Employee(ABC):
    def __init__(self, name):
        self.name = name

    @abstractmethod
    def calculate_salary(self):
        pass


class Developer(Employee):
    def calculate_salary(self):
        return 60000


class Manager(Employee):
    def calculate_salary(self):
        return 90000


employees = [
    Developer("Komal"),
    Manager("Rahul")
]

for employee in employees:
    print(employee.name, employee.calculate_salary())

This combines all four major OOP ideas:

```text
Inheritance
Polymorphism
Encapsulation
Abstraction
```

# 35. Common Mistakes

### Mistake 1 — Thinking `_name` is truly private

```python
self._balance
```

is a convention, not strict access control.

### Mistake 2 — Thinking `__name` is completely inaccessible

Name mangling can technically expose the attribute through a name such as:

```python
_object._ClassName__name
```

### Mistake 3 — Creating a formal abstract class without `ABC`

Use:

```python
class Shape(ABC):
```

and:

```python
@abstractmethod
```

when you want Python's formal abstract-class behavior.

### Mistake 4 — Instantiating an incomplete abstract class

A class with unimplemented abstract methods cannot be instantiated.

### Mistake 5 — Confusing abstraction with hiding everything

Abstraction means exposing a useful interface while hiding unnecessary implementation details.

### Mistake 6 — Making every attribute private

Python encourages sensible interfaces rather than blindly making everything private.

# 36. Final Summary

## Encapsulation

```text
Encapsulation
     ↓
Control access to internal state
     ↓
_conventions
__name / name mangling
@property
methods
validation
```

## Abstraction

```text
Abstraction
     ↓
Hide implementation details
     ↓
ABC
abstractmethod
common interfaces
```

## Connecting the OOP Concepts

```text
                 OOP
                  │
        ┌─────────┼─────────┐
        ↓         ↓         ↓
 Inheritance  Encapsulation  Abstraction
        ↓         ↓         ↓
        └──────→ Polymorphism ←──────┘
```

More practically:

```text
Inheritance
→ Reuse / extend

Polymorphism
→ Same interface, different behavior

Encapsulation
→ Control internal state

Abstraction
→ Hide unnecessary implementation
```